# ADAG1 — Notebook de evaluación

Este notebook carga automáticamente `entrega.json`, muestra la configuración reportada y la evalúa. No realiza una nueva búsqueda ni modifica la configuración. En el entorno del curso, el docente puede conectar el conjunto de prueba reservado; mientras tanto, se incluye una validación pública como comprobación reproducible.


In [ ]:
from pathlib import Path
import json

ENTREGA = Path("entrega.json")
if not ENTREGA.exists():
    raise FileNotFoundError("No se encontró entrega.json en el directorio de trabajo")

with ENTREGA.open(encoding="utf-8") as f:
    datos_entrega = json.load(f)

assert isinstance(datos_entrega, dict)
assert datos_entrega.get("grupo") == "G01", "El grupo debe ser G01 para el Equipo 1"
config = datos_entrega["config"]
for llave in ("rol", "estrategia", "formato", "estilo", "cierre", "temperatura"):
    assert llave in config, f"Falta la llave {llave} en entrega.json"

print("Grupo:", datos_entrega["grupo"])
print("Semana académica:", datos_entrega["semana"])
print("Configuración:", config)


## Carga del modelo y datos

La evaluación completa necesita GPU CUDA, Transformers y los verificadores de `open-instruct`, igual que el notebook de búsqueda.


In [ ]:
from ayudas import cargar_modelo, cargar_datos, dividir
from oraculo import Oraculo

MODELO = "qwen17b"
modelo = cargar_modelo(MODELO)
datos = cargar_datos("datos_visibles.json")
busqueda, validacion = dividir(datos)
oraculo = Oraculo(modelo, busqueda, validacion, lote=16, presupuesto=32_000_000)


## Validación pública de comprobación

Esta celda evalúa `entrega.json` sobre la partición pública `validacion`. No reemplaza la evaluación oficial sobre los datos privados de la profesora.


In [ ]:
resultado = oraculo.validar(config, n=len(validacion))
print("Precisión sobre validación pública:", f"{resultado.precision:.2%}")
print("Número de trazas:", len(resultado.trazas or []))


## Resultado

La configuración se considera reproducible si el archivo `entrega.json` coincide con la configuración reportada por el notebook de búsqueda. La precisión pública sirve como evidencia de validación; la nota oficial depende de los datos privados.
